# Audit Constitution
# 0. 介绍

**研究背景**：Agent 会把大模型给出的动作真正交给工具执行，因此组织不仅要规定`允许做什么`，还要能够回答`当时依据哪版规则、由谁授权、最终是否执行`。这些规则会随环境、风险和合规要求变化，必须独立于模型和 Agent 代码进行更新、检查与追溯。

**现存问题**：生产中真实存在的错误基线，是把治理规则写在系统提示词或 Agent Loop 的硬编码 `if` 中，再在工具执行后补一条普通文本日志。策略更新为‘高风险动作必须审批’后，旧代码仍可能继续放行；即使模型正确声明需要审批，Harness 也可能先执行工具再记录结果。事后日志若没有主体、策略版本、决策依据和完整性证明，就无法可靠解释谁按什么规则放行了动作，也无法发现记录是否被修改。

**解决方案**：本 Notebook 将实现一个极简的 Audit Constitution，采用`部署时 Policy-as-Code + 执行前强制求值 + 版本化审批凭证 + 可验篡改审计链`机制：把规则外置为经过 Schema 校验的 YAML Constitution，为策略保存版本、摘要和变更差异；在每次工具调用前由独立策略关口返回 `allow`、`deny` 或 `ask_human`，没有有效授权就不产生环境副作用；再用结构化 JSONL 记录主体、动作参数摘要、策略版本、决策理由、执行结果、Token、成本与延迟，并用前序哈希连接记录。哈希链用于发现篡改，生产系统还应配合签名、远程追加写或 WORM 存储保护日志根。后文将让同一份真实 API 动作提案经过两条路径：基线版本因旧的硬编码规则错误执行高风险动作，改进版本在加载新策略后改为等待审批、保持环境不变并生成可追溯记录，最后主动修改一条日志验证完整性检查能够报警，从而直观看到可靠治理的关键不是让模型‘自觉守规矩’，而是让外层 Harness 在动作发生前强制执行可版本化的规则，并留下可验证证据。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定生产发布任务
为了只观察治理规则带来的差异，基线版本和改进版本必须处理完全相同的任务。下面固定一次生产服务发布请求；大模型只负责提出动作，是否执行由后续 Harness 决定。

In [2]:
# 这份任务事实会被后续两条实验路径共同使用
# 固定编号和版本可以避免任务变化干扰对照结果
deployment_request = {
    "request_id": "REL-204",
    "service": "payments-prod",
    "current_version": "2.0.0",
    "target_version": "2.1.0",
    "environment": "production",
}

print(deployment_request)

{'request_id': 'REL-204', 'service': 'payments-prod', 'current_version': '2.0.0', 'target_version': '2.1.0', 'environment': 'production'}


输出显示了唯一的发布请求：把生产服务 `payments-prod` 从 `2.0.0` 升级到 `2.1.0`。后续不会更换任务，下一步固定发布发生前的环境状态。

## 2.2 固定环境初态
治理是否有效，最终要看真实状态有没有被越权改变。下面记录发布前的服务版本；后续两条路径都会从这份相同初态开始。

In [3]:
# 初态只保留判断发布是否发生所需的最少字段
# 两条实验路径会各自复制这份数据，互不影响
initial_state = {
    "service": deployment_request["service"],
    "version": deployment_request["current_version"],
}

print(initial_state)

{'service': 'payments-prod', 'version': '2.0.0'}


输出中的版本仍是 `2.0.0`，说明生产发布尚未发生。下一步向大模型说明它应如何提交一张结构化发布提案。

## 2.3 定义模型输出格式
大模型不能直接修改生产状态，只能通过 `propose_deployment` 提交发布提案。下面用 JSON Schema 固定提案字段，并要求模型明确标记生产发布需要人工审批。

In [4]:
# 工具只描述模型应提交的数据，不负责执行生产发布
# requires_approval 让模型对动作风险给出明确判断
tools = [{
    "type": "function",
    "function": {
        "name": "propose_deployment",
        "description": "提出一次生产发布，最终是否执行由外层治理规则决定",
        "parameters": {
            "type": "object",
            "properties": {
                "request_id": {"type": "string"},
                "service": {"type": "string"},
                "target_version": {"type": "string"},
                "environment": {"type": "string"},
                "requires_approval": {"type": "boolean"},
            },
            "required": [
                "request_id",
                "service",
                "target_version",
                "environment",
                "requires_approval",
            ],
        },
    },
}]

print("工具名称：", tools[0]["function"]["name"])
print("必填字段：", tools[0]["function"]["parameters"]["required"])

工具名称： propose_deployment
必填字段： ['request_id', 'service', 'target_version', 'environment', 'requires_approval']


输出列出了模型必须填写的五个字段。这个工具只收集动作提案，不会改变服务版本；下一步把固定任务写成真实 API 能够接收的消息。

## 2.4 准备真实 API 消息
为了把模型判断与 Harness 决策分开，系统消息要求大模型只提交提案，不直接声称发布已经完成。用户消息只包含第 2.1 节固定的发布事实。

In [5]:
# 系统消息限制模型只提出动作，不决定是否放行
# 用户消息直接来自固定任务，后续只发送这一份上下文
messages = [
    {
        "role": "system",
        "content": "你是发布助手。必须调用 propose_deployment 提交提案，不要声称发布已完成；生产发布需要人工审批。",
    },
    {
        "role": "user",
        "content": (
            f"处理 {deployment_request['request_id']}："
            f"将 {deployment_request['service']} "
            f"发布到 {deployment_request['target_version']}。"
        ),
    },
]

print("消息数量：", len(messages))
print("用户任务：", messages[1]["content"])

消息数量： 2
用户任务： 处理 REL-204：将 payments-prod 发布到 2.1.0。


输出显示真实 API 将收到一条系统消息和一条用户消息。此时仍未发送请求；下一步固定改进版本必须达到的结果，避免运行后再改变判断标准。

## 2.5 固定成功标准
当前规则要求生产发布先经过人工审批，所以可靠 Harness 必须在执行前返回 `ask_human`，保持服务版本不变，并把本次决定绑定到可追溯的策略版本和审计记录。

In [6]:
# 这些字段组成后续改进路径唯一的成功结果
# 环境版本保持不变，说明审批前没有执行生产发布
success_criteria = {
    "decision": "ask_human",
    "environment_version": deployment_request["current_version"],
    "approval_request": deployment_request["request_id"],
    "policy_version": "prod-v2",
    "audit_chain_valid": True,
}

print(success_criteria)

{'decision': 'ask_human', 'environment_version': '2.0.0', 'approval_request': 'REL-204', 'policy_version': 'prod-v2', 'audit_chain_valid': True}


输出给出了唯一成功标准：等待人工审批、生产版本仍为 `2.0.0`、审批请求和 `prod-v2` 策略可追溯，并且审计链完整。至此任务、环境、模型协议、消息和判定标准都已固定；下一章才会发送真实 API 请求。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
任务和工具格式已经固定。下面把它们发送给 `.env` 指定的真实大模型，并记录等待时间；模型只会返回发布提案，此时不会改变服务版本。

In [7]:
from time import perf_counter

# 计时范围只覆盖这一次真实 API 请求
# tool_choice 要求模型必须提交结构化发布提案
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)

print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回，但生产发布仍未执行。下一步查看回复中是否包含可以交给 Harness 处理的工具请求。

## 3.2 查看响应类型
一次模型回复可以包含普通文字，也可以包含结构化工具请求。下面分别显示两部分，让读者直接看到模型提交了什么。

In [8]:
# assistant_message 保存模型本次返回的完整消息
# tool_calls 是后续 Harness 真正需要处理的动作提案
assistant_message = response.choices[0].message

print("文字内容：", assistant_message.content)
print("工具请求数量：", len(assistant_message.tool_calls))

文字内容： 我来为您处理 REL-204 的发布请求。
工具请求数量： 1


工具请求数量应为 `1`，表示模型提交了一张结构化发布提案。普通文字不负责改变环境；下一步取出工具名称、调用编号和参数。

## 3.3 保存发布提案
后续基线与改进版本必须共享同一份模型结果。下面把第一个工具请求解析成普通 Python 数据，并保存调用编号，确保之后只比较 Harness 的处理方式。

In [9]:
from json import loads

# 工具请求来自本次真实模型响应
# JSON 参数解析后可直接交给两条 Harness 路径使用
tool_call = assistant_message.tool_calls[0]
call_id = tool_call.id
tool_name = tool_call.function.name
deployment_proposal = loads(tool_call.function.arguments)

print("调用编号：", call_id)
print("工具名称：", tool_name)
print("发布提案：", deployment_proposal)

调用编号： call_0c1de3f30b134172ae8dfd05
工具名称： propose_deployment
发布提案： {'request_id': 'REL-204', 'service': 'payments-prod', 'target_version': '2.1.0', 'environment': 'production', 'requires_approval': True}


输出中的提案应指向 `REL-204`、`payments-prod`、`2.1.0` 和生产环境，并把 `requires_approval` 标记为 `True`。这份结果将在后续两条路径中保持不变；下一步记录本次 API 调用的运行信息。

## 3.4 查看本次请求信息
模型提案已经保存。下面集中显示 provider、模型、停止原因、Token、等待时间和成本状态，使这次真实调用可以被直接核对。

In [10]:
# usage 是真实 provider 随响应返回的 Token 统计
# 项目没有配置模型单价，因此成本明确保留为 None
choice = response.choices[0]
usage = response.usage
api_cost_usd = None

print("Provider：", config["NANO_BACKEND"])
print("Model：", model_name)
print("停止原因：", choice.finish_reason)
print("输入 Token：", usage.prompt_tokens)
print("输出 Token：", usage.completion_tokens)
print("总 Token：", usage.total_tokens)
print("等待时间：", api_latency_ms, "ms")
print("成本：", api_cost_usd)

Provider： openai
Model： LongCat-2.0
停止原因： tool_calls
输入 Token： 246
输出 Token： 227
总 Token： 473
等待时间： 6321 ms
成本： None


输出记录了这次真实请求的来源、停止原因、Token 和延迟；`成本：None` 表示未配置模型单价，而不是估算为零。停止原因 `tool_calls` 只说明模型正在等待 Harness 处理提案，不代表发布已经完成；下一章将定义会错误放行提案的基线组件。

# 4. 定义基线组件
## 4.1 定义生产发布工具
底层工具只负责执行已经获准的动作，不负责制定治理规则。下面定义一个最小发布工具：接收环境状态和发布提案，把当前版本改成目标版本，并返回变化前后的版本。

In [11]:
# 工具只完成版本更新，不判断动作是否应该执行
# 返回前后版本，让环境副作用可以直接观察
def deploy_release(state, proposal):
    before_version = state["version"]
    state["version"] = proposal["target_version"]
    return {"before": before_version, "after": state["version"]}

print("deploy_release 已定义；环境尚未改变")

deploy_release 已定义；环境尚未改变


输出说明发布工具已经定义，但尚未调用，所以生产版本仍是 `2.0.0`。下一步定义会错误放行这项工具调用的旧 Harness。

## 4.2 定义硬编码放行的旧 Harness
生产中常见的错误基线，是把一条旧规则直接写进执行代码。下面的 Harness 无论提案内容是什么都设为 `allow`，随后立即执行发布，只保留一句无法说明授权依据的普通文本。

In [12]:
# 旧规则被写死为 allow，没有读取当前治理策略
# 执行后只留下普通文本，无法绑定策略版本和授权人
def run_legacy_harness(state, proposal):
    decision = "allow"
    tool_result = deploy_release(state, proposal)
    log = f"{proposal['request_id']} deployed"
    return {"decision": decision, "tool_result": tool_result, "log": log}

print("run_legacy_harness 已定义；发布提案尚未执行")

run_legacy_harness 已定义；发布提案尚未执行


输出说明旧 Harness 已经定义，但此时仍没有环境副作用。它没有读取策略文件，也没有处理模型给出的审批标记；下一章将运行它并观察实际故障。

# 5. 展示基线故障
## 5.1 准备基线环境
为了不影响后续改进实验，基线版本使用第 2 章初态的一份独立副本。下面只复制环境，不执行任何发布。

In [13]:
# 基线环境从相同的 2.0.0 版本开始
# 使用副本可以让后续改进路径重新使用原始初态
baseline_state = initial_state.copy()

print("基线环境：", baseline_state)

基线环境： {'service': 'payments-prod', 'version': '2.0.0'}


输出确认基线环境的当前版本是 `2.0.0`，与第 2 章完全一致。下一步把第 3 章保存的同一份真实模型提案交给旧 Harness。

## 5.2 运行旧 Harness
现在调用第 4 章定义的错误基线。它会读取真实模型提案，却不读取当前策略，也不等待人工审批，而是直接执行生产发布。

In [14]:
# 输入是第 3 章保存的真实模型提案
# 旧 Harness 会立即修改 baseline_state
baseline_result = run_legacy_harness(baseline_state, deployment_proposal)

print("旧 Harness 已运行")
print("决策：", baseline_result["decision"])

旧 Harness 已运行
决策： allow


输出中的 `allow` 来自旧 Harness 的硬编码规则，不是模型的授权。发布工具已经被调用；下一步把模型判断、环境变化和日志放在一起查看。

## 5.3 查看故障结果
下面汇总最少的关键状态。重点比较模型是否要求审批、Harness 是否仍然放行，以及生产版本是否已经在没有审批产物的情况下发生变化。

In [15]:
from json import dumps

# 这些字段把模型判断、Harness 决策和环境副作用放在一起
# None 表示旧流程没有生成结构化审批产物
baseline_observation = {
    "model_requires_approval": deployment_proposal["requires_approval"],
    "harness_decision": baseline_result["decision"],
    "version_before": initial_state["version"],
    "version_after": baseline_state["version"],
    "approval_artifact": None,
    "plain_log": baseline_result["log"],
}

print(dumps(baseline_observation, ensure_ascii=False, indent=2))

{
  "model_requires_approval": true,
  "harness_decision": "allow",
  "version_before": "2.0.0",
  "version_after": "2.1.0",
  "approval_artifact": null,
  "plain_log": "REL-204 deployed"
}


结果直接复现了错误基线：模型明确给出 `requires_approval: true`，旧 Harness 却返回 `allow`，生产版本从 `2.0.0` 变成 `2.1.0`；没有审批产物，文本日志也没有策略版本、决策依据或授权主体。问题不在模型和发布工具，而在写死规则并先执行后记录的 Harness。下一章将把规则移出代码，定义可版本化的 Constitution 与结构化审计组件。

# 6. 定义改进组件
## 6.1 加载版本化 Constitution
截至 2026 年 8 月，更可靠的生产做法是把治理规则作为部署时的 Policy-as-Code，由 Harness 在工具执行前读取和求值。下面用最短的 YAML 表示旧规则与当前规则，让策略更新不再依赖修改 Agent Loop。

In [16]:
import yaml

# 两版策略使用相同结构，只有版本和生产发布规则不同
# YAML 让规则可以独立阅读、版本管理和变更审查
legacy_constitution_yaml = """
version: prod-v1
rules:
  propose_deployment:
    production:
      decision: allow
      reason: 旧规则允许自动发布
"""
current_constitution_yaml = """
version: prod-v2
rules:
  propose_deployment:
    production:
      decision: ask_human
      reason: 生产发布必须先由人工审批
"""
legacy_constitution = yaml.safe_load(legacy_constitution_yaml)
current_constitution = yaml.safe_load(current_constitution_yaml)

print("旧策略版本：", legacy_constitution["version"])
print("当前策略版本：", current_constitution["version"])

旧策略版本： prod-v1
当前策略版本： prod-v2


输出显示策略已从 `prod-v1` 更新为 `prod-v2`。两份 YAML 已经成为独立于执行代码的数据；下一步直接展示这次更新改变了什么。

## 6.2 显示策略差异
版本号本身不能说明治理行为。下面把旧版和当前版的生产发布决策并排保存，使审查者一眼看出这次策略更新会怎样改变同一动作。

In [17]:
# 先取出两版策略中的同一条生产发布规则
# diff 只保留真正影响本次实验的版本和决策
legacy_rule = legacy_constitution["rules"]["propose_deployment"]["production"]
current_rule = current_constitution["rules"]["propose_deployment"]["production"]
policy_diff = {
    "version": {
        "before": legacy_constitution["version"],
        "after": current_constitution["version"],
    },
    "production_decision": {
        "before": legacy_rule["decision"],
        "after": current_rule["decision"],
    },
}

print(dumps(policy_diff, ensure_ascii=False, indent=2))

{
  "version": {
    "before": "prod-v1",
    "after": "prod-v2"
  },
  "production_decision": {
    "before": "allow",
    "after": "ask_human"
  }
}


输出明确显示决策从 `allow` 变成 `ask_human`。这正是硬编码基线遗漏的策略更新；下一步为两份原始 YAML 计算摘要，让后续记录能够绑定到确切内容。

## 6.3 计算策略摘要
只记录版本名仍可能指向内容不同的文件。下面对两份 YAML 原文计算 SHA-256 摘要，并把摘要放进已经加载的策略；后续审计记录会同时保存版本与摘要。

In [18]:
from hashlib import sha256

# 摘要由 YAML 原文计算，相同内容会得到相同结果
# 把摘要附在策略对象上，方便后续决策和审计共同引用
legacy_constitution["digest"] = sha256(legacy_constitution_yaml.encode()).hexdigest()
current_constitution["digest"] = sha256(current_constitution_yaml.encode()).hexdigest()

print("旧策略摘要：", legacy_constitution["digest"])
print("当前策略摘要：", current_constitution["digest"] )

旧策略摘要： 2953cf97fc03f3e24c2f3351beaf3d28c28b4bcabd24000f80d7ff0d3b27effe
当前策略摘要： 0649341112590259f232e3aac13eb6afd1ffa9310996fb38bd19a682a5f71464


两个不同摘要说明两版 YAML 内容确实不同。后续只要同时保存 `prod-v2` 和当前摘要，就能明确决策依据的是哪一份策略内容；下一步定义执行前求值器。

## 6.4 定义执行前策略求值器
策略必须在工具产生副作用之前生效。下面的求值器根据工具名称和运行环境读取对应规则，返回决策、理由、策略版本和摘要，但不调用发布工具。

In [19]:
# 工具名称和环境共同定位本次应使用的规则
# 返回结果携带完整策略身份，供执行关口和审计共同使用
def evaluate_constitution(policy, action_name, proposal):
    environment = proposal["environment"]
    rule = policy["rules"][action_name][environment]
    return {
        "decision": rule["decision"],
        "reason": rule["reason"],
        "policy_version": policy["version"],
        "policy_hash": policy["digest"],
    }

print("evaluate_constitution 已定义；当前策略尚未求值")

evaluate_constitution 已定义；当前策略尚未求值


输出说明求值器已定义，但当前策略尚未处理提案。它只负责回答是否允许，不会越过边界执行动作；下一步定义 `ask_human` 决策对应的审批产物。

## 6.5 定义审批产物
`ask_human` 不能只停留在一行提示文字中。下面把等待审批变成结构化数据，保存请求编号、状态、原因以及策略身份，让后续人工流程和审计能够引用同一份凭证。

In [20]:
# 审批产物使用任务编号连接原始发布请求
# 策略版本和摘要说明为什么这次动作需要等待
def create_approval_artifact(proposal, decision):
    return {
        "request_id": proposal["request_id"],
        "status": "pending",
        "reason": decision["reason"],
        "policy_version": decision["policy_version"],
        "policy_hash": decision["policy_hash"],
    }

print("create_approval_artifact 已定义；审批产物尚未生成")

create_approval_artifact 已定义；审批产物尚未生成


输出说明审批产物函数已经定义，但尚未创建具体凭证。下一步定义唯一可以连接策略决策与发布工具的执行关口。

## 6.6 定义执行关口
执行关口是改进方案最关键的控制点：`allow` 才调用发布工具，`ask_human` 只生成审批产物。下面把这两条路径写成显式分支，使决策与环境副作用的关系清楚可见。

In [21]:
# 默认没有工具结果，也没有审批产物
# 只有策略明确返回 allow 时才会修改环境状态
def apply_policy_decision(state, proposal, decision):
    tool_result = None
    approval_artifact = None

    if decision["decision"] == "allow":
        tool_result = deploy_release(state, proposal)

    if decision["decision"] == "ask_human":
        approval_artifact = create_approval_artifact(proposal, decision)

    return {"tool_result": tool_result, "approval_artifact": approval_artifact}

print("apply_policy_decision 已定义；当前提案尚未进入关口")

apply_policy_decision 已定义；当前提案尚未进入关口


输出说明执行关口已经定义，但尚未处理提案。代码中只有 `allow` 分支能够调用 `deploy_release`；当前 `ask_human` 规则将只产生审批数据。下一步定义结构化哈希链审计。

## 6.7 定义哈希链审计
普通文本日志无法证明记录顺序和内容是否被改动。下面的追加器先写入前一条记录的哈希，再为当前结构化事件计算哈希；每条记录因此都会连接到前一条记录。

In [22]:
# 第一条记录连接固定起点，后续记录连接前一条哈希
# 当前哈希覆盖事件内容和 previous_hash，形成可追溯顺序
def append_audit(audit_log, event):
    record = event.copy()

    if audit_log:
        record["previous_hash"] = audit_log[-1]["hash"]
    else:
        record["previous_hash"] = "GENESIS"

    payload = dumps(record, ensure_ascii=False, sort_keys=True)
    record["hash"] = sha256(payload.encode()).hexdigest()
    audit_log.append(record)
    return record

print("append_audit 已定义；审计链尚未创建")

append_audit 已定义；审计链尚未创建


输出说明审计追加器已经定义，但目前没有审计记录。本章至此完成了版本化 YAML、策略差异、策略摘要、执行前求值、审批产物、执行关口和哈希链七个核心构件；下一章将用当前策略运行同一份真实模型提案。

# 7. 展示修复结果
## 7.1 准备改进环境
改进版本必须与基线从同一状态开始。下面重新复制第 2 章的环境初态，并准备一条空审计链；此时没有执行发布，也没有审计记录。

In [23]:
# 改进环境重新从 2.0.0 开始，不复用已污染的基线状态
# 空列表将按发生顺序保存本次治理事件
fixed_state = initial_state.copy()
fixed_audit_log = []

print("改进环境：", fixed_state)
print("审计记录数量：", len(fixed_audit_log))

改进环境： {'service': 'payments-prod', 'version': '2.0.0'}
审计记录数量： 0


输出确认改进环境仍为 `2.0.0`，审计链为空。下一步让 `prod-v2` Constitution 在任何工具副作用发生前处理真实模型提案。

## 7.2 求值当前策略
下面把第 3 章保存的同一份提案交给当前 Constitution。求值器只读取工具名称和生产环境，返回治理决策及其完整策略身份。

In [24]:
# 输入仍是同一个真实模型提案和 propose_deployment 工具名
# current_constitution 是第 6 章加载的 prod-v2 策略
fixed_decision = evaluate_constitution(
    current_constitution,
    tool_name,
    deployment_proposal,
)

print(dumps(fixed_decision, ensure_ascii=False, indent=2))

{
  "decision": "ask_human",
  "reason": "生产发布必须先由人工审批",
  "policy_version": "prod-v2",
  "policy_hash": "0649341112590259f232e3aac13eb6afd1ffa9310996fb38bd19a682a5f71464"
}


输出中的 `ask_human` 来自 `prod-v2`，并带有规则理由与策略摘要。此时发布工具仍未执行；下一步把这项决策交给执行关口。

## 7.3 通过执行关口
执行关口现在处理 `ask_human`：它不会调用发布工具，只会创建待审批产物。下面执行这一步，并同时显示工具结果、审批产物和环境状态。

In [25]:
# 执行关口是治理决策与环境副作用之间的唯一连接点
# ask_human 路径应返回审批产物，并让 tool_result 保持 None
fixed_result = apply_policy_decision(
    fixed_state,
    deployment_proposal,
    fixed_decision,
)

print("工具结果：", fixed_result["tool_result"])
print("审批产物：", fixed_result["approval_artifact"])
print("环境状态：", fixed_state)

工具结果： None
审批产物： {'request_id': 'REL-204', 'status': 'pending', 'reason': '生产发布必须先由人工审批', 'policy_version': 'prod-v2', 'policy_hash': '0649341112590259f232e3aac13eb6afd1ffa9310996fb38bd19a682a5f71464'}
环境状态： {'service': 'payments-prod', 'version': '2.0.0'}


`工具结果：None` 表示发布工具没有执行，环境版本仍为 `2.0.0`；审批产物则明确处于 `pending`，并绑定 `prod-v2` 及其摘要。下一步把这项策略决策写入审计链。

## 7.4 记录策略决策
第一条审计记录需要回答谁提出了什么动作、参数是什么、依据哪份策略以及为何等待审批。下面对提案参数计算摘要，再把完整决策事件追加到空审计链。

In [26]:
# 参数摘要避免在索引字段中重复整份提案，同时保持内容可关联
# 第一条记录的 previous_hash 将由追加器设为 GENESIS
proposal_text = dumps(deployment_proposal, ensure_ascii=False, sort_keys=True)
proposal_hash = sha256(proposal_text.encode()).hexdigest()
decision_event = {
    "trace_id": "trace-REL-204",
    "event": "policy_decision",
    "principal": "release-agent",
    "tool": tool_name,
    "arguments_hash": proposal_hash,
    "decision": fixed_decision["decision"],
    "reason": fixed_decision["reason"],
    "policy_version": fixed_decision["policy_version"],
    "policy_hash": fixed_decision["policy_hash"],
    "result": "waiting_approval",
}
decision_audit_record = append_audit(fixed_audit_log, decision_event)

print(dumps(decision_audit_record, ensure_ascii=False, indent=2))

{
  "trace_id": "trace-REL-204",
  "event": "policy_decision",
  "principal": "release-agent",
  "tool": "propose_deployment",
  "arguments_hash": "9dac43d1807a000b2cfbed02b8dec3fd7c4d17a374398a22dc95c399fd60b9df",
  "decision": "ask_human",
  "reason": "生产发布必须先由人工审批",
  "policy_version": "prod-v2",
  "policy_hash": "0649341112590259f232e3aac13eb6afd1ffa9310996fb38bd19a682a5f71464",
  "result": "waiting_approval",
  "previous_hash": "GENESIS",
  "hash": "c08180d2399172d4630abc9488d934f6150d18ccb7ad4a140eb0edb3cc2b2582"
}


第一条记录已经包含 trace、主体、工具、参数摘要、决策、理由、策略版本、策略摘要和结果，并从 `GENESIS` 开始。下一步把实际生成的审批产物作为第二个事件接到它后面。

## 7.5 记录审批产物
第二条记录说明治理决策产生了什么后续结果。下面保存审批请求的编号、状态和策略版本；追加器会自动让它的 `previous_hash` 指向第一条记录。

In [27]:
# 第二个事件引用执行关口刚刚生成的审批产物
# append_audit 会把它连接到上一条策略决策记录
approval_event = {
    "trace_id": "trace-REL-204",
    "event": "approval_created",
    "principal": "release-agent",
    "request_id": fixed_result["approval_artifact"]["request_id"],
    "policy_version": fixed_result["approval_artifact"]["policy_version"],
    "result": fixed_result["approval_artifact"]["status"],
}
approval_audit_record = append_audit(fixed_audit_log, approval_event)

print(dumps(approval_audit_record, ensure_ascii=False, indent=2))

{
  "trace_id": "trace-REL-204",
  "event": "approval_created",
  "principal": "release-agent",
  "request_id": "REL-204",
  "policy_version": "prod-v2",
  "result": "pending",
  "previous_hash": "c08180d2399172d4630abc9488d934f6150d18ccb7ad4a140eb0edb3cc2b2582",
  "hash": "978edda6d4ac9470f562e6b73d59ca68f85ab54bc1469b616d7cc71569720ee7"
}


第二条记录的 `previous_hash` 与第一条记录的 `hash` 相同，因此两个事件已经按顺序连接。下一步汇总环境、审批与审计状态，直接查看修复后的最终结果。

## 7.6 查看修复结果
最后把与第 5 章相同的关键状态放在一起。可靠结果应是模型仍提出同一动作，Harness 改为等待审批，生产版本不变，并留下待审批产物和两条结构化审计记录。

In [28]:
# 汇总字段与基线故障保持同一观察口径
# 审计尾哈希代表当前两条记录形成的完整链尾
fixed_observation = {
    "model_requires_approval": deployment_proposal["requires_approval"],
    "harness_decision": fixed_decision["decision"],
    "version_before": initial_state["version"],
    "version_after": fixed_state["version"],
    "tool_result": fixed_result["tool_result"],
    "approval_status": fixed_result["approval_artifact"]["status"],
    "policy_version": fixed_decision["policy_version"],
    "audit_records": len(fixed_audit_log),
    "audit_tail_hash": fixed_audit_log[-1]["hash"],
}

print(dumps(fixed_observation, ensure_ascii=False, indent=2))

{
  "model_requires_approval": true,
  "harness_decision": "ask_human",
  "version_before": "2.0.0",
  "version_after": "2.0.0",
  "tool_result": null,
  "approval_status": "pending",
  "policy_version": "prod-v2",
  "audit_records": 2,
  "audit_tail_hash": "978edda6d4ac9470f562e6b73d59ca68f85ab54bc1469b616d7cc71569720ee7"
}


结果表明修复已经生效：同一份真实模型提案被 `prod-v2` 判定为 `ask_human`，发布工具没有执行，生产版本保持 `2.0.0`，审批状态为 `pending`，并生成两条相连的审计记录。变化只来自外层 Harness 采用了当前 Constitution；下一章将把基线与改进结果汇总为最终消融对照。

# 8. 汇总消融对照
## 8.1 对比基线与改进结果
两条路径复用了第 3 章的同一份真实模型提案，没有增加模型请求。下面把决策、环境终态、审批和审计放在同一份结构化结果中，使 Harness 构件带来的差异可以直接比较。

In [29]:
# 两条路径共享同一模型提案，因此模型 Token、成本和延迟增量都为零
# 对照只改变外层是否加载 Constitution、执行关口和结构化审计
ablation_summary = {
    "shared_model_input": {
        "request_id": deployment_proposal["request_id"],
        "requires_approval": deployment_proposal["requires_approval"],
        "extra_api_calls": 0,
        "token_delta": 0,
        "cost_delta_usd": 0,
        "latency_delta_ms": 0,
    },
    "hardcoded_baseline": {
        "decision": baseline_observation["harness_decision"],
        "version_after": baseline_observation["version_after"],
        "approval": baseline_observation["approval_artifact"],
        "audit": baseline_observation["plain_log"],
    },
    "constitution_audit": {
        "decision": fixed_observation["harness_decision"],
        "version_after": fixed_observation["version_after"],
        "approval": fixed_observation["approval_status"],
        "policy_version": fixed_observation["policy_version"],
        "audit_records": fixed_observation["audit_records"],
    },
}

print(dumps(ablation_summary, ensure_ascii=False, indent=2))

{
  "shared_model_input": {
    "request_id": "REL-204",
    "requires_approval": true,
    "extra_api_calls": 0,
    "token_delta": 0,
    "cost_delta_usd": 0,
    "latency_delta_ms": 0
  },
  "hardcoded_baseline": {
    "decision": "allow",
    "version_after": "2.1.0",
    "approval": null,
    "audit": "REL-204 deployed"
  },
  "constitution_audit": {
    "decision": "ask_human",
    "version_after": "2.0.0",
    "approval": "pending",
    "policy_version": "prod-v2",
    "audit_records": 2
  }
}


对照结果表明，模型输入和 API 开销没有变化；硬编码基线直接把版本改成 `2.1.0`，改进路径则返回 `ask_human`、保持 `2.0.0`、生成待审批产物和两条审计记录。可靠性差异完全来自模型外层的治理机制。下一步补上读取哈希链时所需的最小哈希计算。

## 8.2 定义审计哈希计算
检查一条审计记录时，需要先移除其中已经保存的 `hash`，再对其余字段按与写入时相同的方式计算摘要。下面把这一步写成一个最小函数。

In [30]:
# 复制记录可以避免计算过程改动原始审计链
# 排序后的 JSON 与 append_audit 使用完全相同的哈希输入
def calculate_audit_hash(record):
    content = record.copy()
    content.pop("hash")
    payload = dumps(content, ensure_ascii=False, sort_keys=True)
    return sha256(payload.encode()).hexdigest()

print("calculate_audit_hash 已定义；原始审计链未改变")

calculate_audit_hash 已定义；原始审计链未改变


输出说明哈希计算函数已经定义，原始审计链仍保持第 7 章的内容。下一步只修改第一条记录的副本，观察保存的哈希是否还能匹配。

## 8.3 展示篡改可见性
为了直接说明完整性哈希的作用，下面把第一条审计记录复制一份，并把副本中的决策从 `ask_human` 改成 `allow`。然后分别重算原记录和修改后副本的哈希。

In [31]:
# 只修改副本，fixed_audit_log 中的原始记录保持不变
# 修改任一受哈希覆盖的字段都会产生不同的重算结果
tampered_record = fixed_audit_log[0].copy()
tampered_record["decision"] = "allow"
stored_hash = fixed_audit_log[0]["hash"]
original_hash = calculate_audit_hash(fixed_audit_log[0])
tampered_hash = calculate_audit_hash(tampered_record)
integrity_demo = {
    "stored_hash": stored_hash,
    "original_matches": original_hash == stored_hash,
    "tampered_matches": tampered_hash == stored_hash,
    "tampering_detected": tampered_hash != stored_hash,
}

print(dumps(integrity_demo, ensure_ascii=False, indent=2))

{
  "stored_hash": "c08180d2399172d4630abc9488d934f6150d18ccb7ad4a140eb0edb3cc2b2582",
  "original_matches": true,
  "tampered_matches": false,
  "tampering_detected": true
}


原记录重算后仍匹配保存值，修改过决策的副本则不再匹配，因此 `tampering_detected` 为 `true`。这说明哈希链能暴露记录被改动，但不能阻止文件被改写；生产系统还需要签名、远程追加写或 WORM 存储保护可信日志根。

## 8.4 拓展
### nano 版省略了什么
生产系统通常还需要标准化策略 Schema、OPA/Rego、Cedar、Progent 或 AgentSpec 等更强策略语言，策略冲突与覆盖检查，组织身份和委托链，真实审批界面，签名与密钥管理，远程日志锚定、保留和隐私规则，以及跨多步轨迹的审计分析。本 Notebook 只保留理解声明式策略、执行关口和可验篡改审计链所需的最短闭环。

### 延伸阅读


1. 2026, [Anthropic, Claude's Constitution](https://www.anthropic.com/constitution)：公开、可版本化的行为原则及其治理定位。
2. 2024, [NIST AI 600-1, Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)：生成式 AI 风险识别、测量、治理与记录要求。
3. 2024, [EU Artificial Intelligence Act](https://eur-lex.europa.eu/eli/reg/2024/1689/oj)：高风险系统的日志、监督、透明度与责任要求。